# Step 2 — BGE-M3 + TF-IDF Representation

**What changed vs. the version you gave me, and why:**

1. **TF-IDF was described but never actually built.** The original script only saved raw `combined_text`,
   not a TF-IDF vector — this notebook fits real word-level *and* character-level TF-IDF vectorizers
   (char n-grams catch typos/transliteration noise that word n-grams miss entirely) and saves the
   resulting sparse vectors alongside the embeddings.
2. **TF-IDF is fit once, shared across all three sources.** Fitting a separate vectorizer per source
   (or per chunk) gives each one a different vocabulary/IDF weighting — cosine similarity between a
   Source 1 TF-IDF vector and a Source 2 TF-IDF vector would then be meaningless, since they wouldn't
   live in the same vector space. The vectorizer is saved once under `processed/shared/` and reused.
3. **The literal `"[SEP]"` string is not a special token BGE-M3's tokenizer recognizes** (BGE-M3 is built
   on an XLM-RoBERTa backbone, which uses `<s>`/`</s>`, not BERT-style `[SEP]`) — as written, it would just
   get tokenized as ordinary sub-word pieces (literal brackets + letters), adding noise rather than acting
   as a separator. Replaced with a plain `" | "` delimiter, which costs nothing and doesn't rely on the
   model treating a string as something it wasn't trained to.
4. **GPU/fp16 handling.** `use_fp16=True` unconditionally is a no-op-or-worse on a CPU-only box. Now
   detected automatically.
5. **Embeddings are explicitly L2-normalized and cast to `float32`** before saving — needed so a FAISS
   inner-product index gives you exact cosine similarity later, and so you're not silently storing
   double the memory in `float64`.
6. **Resumability preserved** (skip already-completed chunks), same as your original design.

Run the cells top to bottom. The TF-IDF fitting cell only needs to run once per project — see the
warning in that cell before re-running it for source2/source3.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as l2_normalize
from FlagEmbedding import BGEM3FlagModel
from tqdm.auto import tqdm

In [ ]:
# ============================================================
# CONFIG
# ============================================================

# Change this to "source2" / "source3" when you run this notebook for
# the other sources. Everything below derives from it.
SOURCE_LABEL = "source1"

INPUT_DIR = Path(f"processed/{SOURCE_LABEL}_cleaned")
OUTPUT_DIR = Path(f"processed/{SOURCE_LABEL}_embeddings")

# TF-IDF vectorizers live here - shared across source1/2/3, fit once.
SHARED_DIR = Path("processed/shared")

# All three sources' cleaned dirs, used only for sampling text to fit
# the shared TF-IDF vectorizer (see the warning in the fitting cell).
ALL_CLEANED_DIRS = [
    Path("processed/source1_cleaned"),
    Path("processed/source2_cleaned"),
    Path("processed/source3_cleaned"),
]

BATCH_SIZE = 256
MAX_LENGTH = 160  # combined name+address text is short; 160 gives headroom over the original 128

TFIDF_FIT_SAMPLE_SIZE = 300_000
TFIDF_WORD_MAX_FEATURES = 200_000
TFIDF_CHAR_MAX_FEATURES = 200_000
USE_CHAR_TFIDF = True  # char n-grams catch typos/transliteration; set False to save time/storage if needed

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHARED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# DEVICE / MODEL SETUP
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"  # fp16 only helps on GPU; on CPU it's a no-op or worse

print(f"Device: {DEVICE} | fp16: {USE_FP16}")

model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=USE_FP16,
)

In [ ]:
# ============================================================
# TEXT COMBINATION
# ============================================================

def build_combined_text(name: str, address: str) -> str:
    """
    Plain ' | ' delimiter instead of the literal string "[SEP]" - see the
    note at the top of this notebook on why "[SEP]" doesn't do what it
    looks like it does for this model.
    """
    name = (name or "").strip()
    address = (address or "").strip()
    if not name and not address:
        return ""
    if not address:
        return name
    if not name:
        return address
    return f"{name} | {address}"

In [ ]:
# ============================================================
# SHARED TF-IDF: FIT ONCE, REUSE ACROSS SOURCES
# ============================================================
# WARNING - read before running:
# This cell fits the shared word- and char-level TF-IDF vectorizers ONLY
# IF they don't already exist under SHARED_DIR. It samples text from
# whichever of source1/2/3 cleaned directories currently exist.
#
# If you're running this for the very first time and only source1 has
# been cleaned so far, the vectorizer will be fit on source1 text alone.
# That's a reasonable starting point, but source2/source3 typically have
# noisier vocabulary (more typos, more abbreviation variants). Once all
# three sources are cleaned, DELETE processed/shared/tfidf_word.joblib
# and processed/shared/tfidf_char.joblib and re-run this cell once so the
# vectorizer reflects the full vocabulary - then re-run the embedding
# step for every source so their TF-IDF vectors all come from the same
# fit. Do NOT refit per source after that; every source's TF-IDF vectors
# must come from the exact same fitted vectorizer or cosine similarity
# between sources becomes meaningless.

WORD_VEC_PATH = SHARED_DIR / "tfidf_word.joblib"
CHAR_VEC_PATH = SHARED_DIR / "tfidf_char.joblib"


def sample_texts_for_tfidf(cleaned_dirs, sample_size, seed=42):
    texts = []
    for d in cleaned_dirs:
        if not d.exists():
            print(f"  (skipping {d}, not cleaned yet)")
            continue
        for f in sorted(d.glob("part_*.parquet")):
            df = pd.read_parquet(f, columns=["name_full_clean", "address_clean"])
            combined = df["name_full_clean"].fillna("") + " | " + df["address_clean"].fillna("")
            texts.extend(combined.tolist())

    if not texts:
        raise RuntimeError(
            "No cleaned parquet files found in any of ALL_CLEANED_DIRS - "
            "run the Step 1 cleaning script first."
        )

    rng = np.random.default_rng(seed)
    if len(texts) > sample_size:
        idx = rng.choice(len(texts), size=sample_size, replace=False)
        texts = [texts[i] for i in idx]

    print(f"Sampled {len(texts):,} texts to fit TF-IDF on.")
    return texts


def fit_or_load_tfidf():
    if WORD_VEC_PATH.exists() and (not USE_CHAR_TFIDF or CHAR_VEC_PATH.exists()):
        print("Loading existing shared TF-IDF vectorizer(s).")
        word_vec = joblib.load(WORD_VEC_PATH)
        char_vec = joblib.load(CHAR_VEC_PATH) if USE_CHAR_TFIDF else None
        return word_vec, char_vec

    print("Fitting shared TF-IDF vectorizer(s) - this should happen once for the whole project.")
    texts = sample_texts_for_tfidf(ALL_CLEANED_DIRS, TFIDF_FIT_SAMPLE_SIZE)

    word_vec = TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2), min_df=2, max_features=TFIDF_WORD_MAX_FEATURES
    )
    word_vec.fit(texts)
    joblib.dump(word_vec, WORD_VEC_PATH)

    char_vec = None
    if USE_CHAR_TFIDF:
        char_vec = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=TFIDF_CHAR_MAX_FEATURES
        )
        char_vec.fit(texts)
        joblib.dump(char_vec, CHAR_VEC_PATH)

    return word_vec, char_vec


word_vectorizer, char_vectorizer = fit_or_load_tfidf()
print("Word vocab size:", len(word_vectorizer.vocabulary_))
if char_vectorizer is not None:
    print("Char vocab size:", len(char_vectorizer.vocabulary_))

In [ ]:
# ============================================================
# PER-CHUNK PROCESSING
# ============================================================

def process_chunk(file: Path):
    stem = file.stem  # e.g. "part_00000"

    meta_out = OUTPUT_DIR / f"{stem}_meta.parquet"
    word_out = OUTPUT_DIR / f"{stem}_tfidf_word.npz"
    char_out = OUTPUT_DIR / f"{stem}_tfidf_char.npz"
    done_file = OUTPUT_DIR / f"{stem}.done"

    if done_file.exists() and meta_out.exists():
        print(f"[SKIP] {stem}")
        return

    print(f"[PROCESS] {stem}")
    df = pd.read_parquet(file, columns=["entity_id", "name_full_clean", "address_clean", "country"])

    df["combined_text"] = [
        build_combined_text(n, a) for n, a in zip(df["name_full_clean"], df["address_clean"])
    ]
    texts = df["combined_text"].tolist()

    # ---- BGE-M3 dense embeddings ----
    embeddings = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=f"{stem} [BGE]"):
        batch = texts[i : i + BATCH_SIZE]
        emb = model.encode(batch, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)["dense_vecs"]
        embeddings.append(np.asarray(emb, dtype=np.float32))

    embeddings = np.vstack(embeddings)
    embeddings = l2_normalize(embeddings, axis=1)  # exact cosine similarity later, regardless of raw output norm
    embeddings = embeddings.astype(np.float32)

    # ---- TF-IDF (word + optional char), reusing the shared fitted vectorizers ----
    word_matrix = word_vectorizer.transform(texts)
    sp.save_npz(word_out, word_matrix.astype(np.float32))

    if char_vectorizer is not None:
        char_matrix = char_vectorizer.transform(texts)
        sp.save_npz(char_out, char_matrix.astype(np.float32))

    # ---- Save light metadata + dense embedding together ----
    df["embedding"] = list(embeddings)
    df[["entity_id", "combined_text", "embedding", "country"]].to_parquet(meta_out, index=False)

    done_file.write_text("DONE\n", encoding="utf-8")
    print(f"[DONE] {stem}")

In [ ]:
# ============================================================
# RUN
# ============================================================

files = sorted(INPUT_DIR.glob("part_*.parquet"))
print(f"Found {len(files)} cleaned parts for {SOURCE_LABEL}")

for f in files:
    process_chunk(f)

(OUTPUT_DIR / "_SUCCESS").touch()
print(f"Step 2 completed for {SOURCE_LABEL}.")

## Optional: consolidate into a single memory-mapped array

The per-chunk `_meta.parquet` files are fine for a few hundred thousand rows, but at 50L+ scale, loading
embeddings out of parquet list-columns to build a FAISS index gets slow. This optional cell concatenates
every chunk's embeddings into one contiguous `float32` `.npy` file plus a matching `entity_id` order file,
which FAISS can load directly and efficiently. Run it only after every chunk above has finished.

In [ ]:
CONSOLIDATE = False  # set True to build the single memmap-friendly array below

if CONSOLIDATE:
    meta_files = sorted(OUTPUT_DIR.glob("part_*_meta.parquet"))

    all_ids = []
    all_embeddings = []

    for f in tqdm(meta_files, desc="Consolidating"):
        df = pd.read_parquet(f, columns=["entity_id", "embedding"])
        all_ids.extend(df["entity_id"].tolist())
        all_embeddings.append(np.vstack(df["embedding"].tolist()).astype(np.float32))

    all_embeddings = np.vstack(all_embeddings)

    np.save(OUTPUT_DIR / "embeddings.npy", all_embeddings)
    with open(OUTPUT_DIR / "entity_id_order.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(all_ids))

    print(f"Saved {all_embeddings.shape} embeddings and matching entity_id order.")